# COPOD — Copula-Based Outlier Detection\nÖzellikler arası bağımlılığı Copula teorisiyle modelleyen olasılıksal anomali tespiti.

In [ ]:
import numpy as np, matplotlib.pyplot as plt\nfrom pyod.models.copod import COPOD\nfrom sklearn.datasets import make_blobs\nimport os; os.makedirs('cikti', exist_ok=True)

## COPOD ile Anomali Tespiti

In [ ]:
X, _ = make_blobs(n_samples=300, n_features=5, centers=1, cluster_std=1.0, random_state=42)\nanomalies = np.random.uniform(low=-8, high=8, size=(20, 5))\nX = np.vstack([X, anomalies])\n\nmodel = COPOD(contamination=0.06)\nmodel.fit(X)\npreds = model.predict(X)\nscores = model.decision_scores_\n\nprint(f'Normal: {(preds == 0).sum()}, Anomali: {(preds == 1).sum()}')\nprint(f'Anomali skor araligi: [{scores.min():.2f}, {scores.max():.2f}]')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))\n\naxes[0].hist(scores[preds == 0], bins=30, alpha=0.6, label='Normal', color='blue')\naxes[0].hist(scores[preds == 1], bins=30, alpha=0.6, label='Anomali', color='red')\naxes[0].axvline(model.threshold_, color='black', linestyle='--', label=f'Eşik ({model.threshold_:.2f})')\naxes[0].set_xlabel('Anomali Skoru'); axes[0].set_title('COPOD Skor Dağılımı')\naxes[0].legend()\n\n# En yüksek 5 skoru göster\ntop5 = np.argsort(scores)[-5:]\nfor idx in top5[::-1]:\n    axes[1].barh(f'Ornek {idx}', scores[idx], color='red' if preds[idx] == 1 else 'blue')\naxes[1].axvline(model.threshold_, color='black', linestyle='--')\naxes[1].set_xlabel('Anomali Skoru'); axes[1].set_title('En Yüksek 5 Anomali Skoru')\nplt.tight_layout(); plt.savefig('cikti/copod_skorlari.png', dpi=100)\nplt.show()

## Isolation Forest / One-Class SVM ile Karşılaştırma

In [ ]:
from sklearn.ensemble import IsolationForest\nfrom sklearn.svm import OneClassSVM\n\n# Aynı veri, 3 farklı algoritma\nalgoritmalar = {\n    'COPOD': COPOD(contamination=0.06),\n    'Isolation Forest': IsolationForest(contamination=0.06, random_state=42),\n    'One-Class SVM': OneClassSVM(nu=0.06),\n}\n\nsonuclar = {}\nfor isim, algo in algoritmalar.items():\n    algo.fit(X)\n    if isim == 'One-Class SVM':\n        p = algo.predict(X)\n        n_anomali = (p == -1).sum()\n    else:\n        p = algo.predict(X)\n        n_anomali = (p == 1).sum() if isim == 'COPOD' else (p == -1).sum()\n    sonuclar[isim] = n_anomali\n    print(f'{isim:<20}: {n_anomali} anomali tespit edildi')\n\n# Görselleştir\nplt.figure(figsize=(8, 4))\ncolors = ['purple', 'green', 'orange']\nfor (isim, n), c in zip(sonuclar.items(), colors):\n    plt.bar(isim, n, color=c, alpha=0.7)\nplt.ylabel('Tespit Edilen Anomali'); plt.title('Algoritma Karşılaştırması')\nplt.savefig('cikti/anomali_karsilastirma.png', dpi=100, bbox_inches='tight')\nplt.show()